# EDA Agent - Ollama on a Kaggle GPU + Cloudflare tunnel

Serves **qwen2.5-coder:14b** (agent) and **qwen2.5:14b-instruct** (judge) on a Kaggle GPU and exposes
them through a Cloudflare *quick tunnel*, so the local Windows machine (mode A: agent + Docker sandbox +
UI run locally) can call the LLM over HTTPS.

**Notebook settings:** Accelerator = GPU T4 x2 (or P100), Internet = ON (needs a phone-verified Kaggle
account), Persistence = none.

Run the cells top to bottom and **leave the last cell running**.
Limits: 12 h per session, about 30 GPU hours per week. If the session stops, the tunnel URL changes:
re-run everything and set the new URL locally. The eval runner resumes from saved results.

**Security:** the `trycloudflare.com` URL is public while the tunnel runs (anyone with the URL can use
this Ollama). Do not share it and stop the session when you are done.

In [ ]:
AGENT_MODEL = "qwen2.5-coder:14b"
JUDGE_MODEL = "qwen2.5:14b-instruct"
PULL_JUDGE = True                  # False if you only need the agent model now
CLOUDFLARED_VERSION = "2024.12.2"  # pinned release of cloudflared
OLLAMA_PORT = 11434
WORK = "/kaggle/working"

import json, os, re, subprocess, time, urllib.request
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv"],
                     capture_output=True, text=True).stdout)

## 1. Install Ollama

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

## 2. Start the Ollama server (keep_alive 24h so the model is never unloaded)

In [ ]:
def start_ollama():
    env = dict(os.environ, OLLAMA_HOST=f"0.0.0.0:{OLLAMA_PORT}", OLLAMA_KEEP_ALIVE="24h",
               OLLAMA_NUM_PARALLEL="1", OLLAMA_MAX_LOADED_MODELS="1")
    log = open(f"{WORK}/ollama.log", "a")
    return subprocess.Popen(["ollama", "serve"], env=env, stdout=log, stderr=subprocess.STDOUT)

def ollama_ready():
    try:
        with urllib.request.urlopen(f"http://localhost:{OLLAMA_PORT}/api/version", timeout=3) as r:
            return json.load(r)
    except Exception:
        return None

ollama_proc = start_ollama()
for _ in range(60):
    if ollama_ready():
        break
    time.sleep(1)
print("ollama:", ollama_ready())

## 3. Pull the models (about 9 GB each, a few minutes)

In [ ]:
for m in [AGENT_MODEL] + ([JUDGE_MODEL] if PULL_JUDGE else []):
    subprocess.run(["ollama", "pull", m], check=True)
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)

## 4. Local test inside Kaggle (loads the model on the GPU, measures speed)

In [ ]:
def chat(model, prompt, num_predict=100):
    body = json.dumps({"model": model, "messages": [{"role": "user", "content": prompt}],
                       "stream": False, "keep_alive": "24h",
                       "options": {"temperature": 0.1, "seed": 42, "num_ctx": 8192,
                                   "num_predict": num_predict}}).encode()
    req = urllib.request.Request(f"http://localhost:{OLLAMA_PORT}/api/chat", data=body,
                                 headers={"Content-Type": "application/json"})
    t0 = time.time()
    with urllib.request.urlopen(req, timeout=900) as r:
        out = json.load(r)
    dt = time.time() - t0
    print(f"{model}: {out.get('eval_count')} tokens in {dt:.1f} s; reply: {out['message']['content'][:80]!r}")

chat(AGENT_MODEL, "Write a Python function that returns the mean of a list. Code only.")
print(subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv"],
                     capture_output=True, text=True).stdout)

## 5. Start the tunnel

`--http-host-header localhost:11434` is required: without it Ollama receives the public host name
and rejects the request (HTTP 403).

In [ ]:
CF = f"{WORK}/cloudflared"
if not os.path.exists(CF):
    subprocess.run(["wget", "-q", "-O", CF,
                    f"https://github.com/cloudflare/cloudflared/releases/download/{CLOUDFLARED_VERSION}/cloudflared-linux-amd64"],
                   check=True)
    os.chmod(CF, 0o755)

URL_RE = re.compile(r"https://[-a-z0-9]+\.trycloudflare\.com")

def start_tunnel():
    log_path = f"{WORK}/cloudflared.log"
    log = open(log_path, "w")
    proc = subprocess.Popen([CF, "tunnel", "--no-autoupdate", "--url", f"http://localhost:{OLLAMA_PORT}",
                             "--http-host-header", f"localhost:{OLLAMA_PORT}"],
                            stdout=log, stderr=subprocess.STDOUT)
    for _ in range(90):
        time.sleep(1)
        m = URL_RE.search(open(log_path).read())
        if m:
            return proc, m.group(0)
    raise RuntimeError("tunnel URL not found; read cloudflared.log")

tunnel_proc, TUNNEL_URL = start_tunnel()
print("TUNNEL URL:", TUNNEL_URL)
print("On Windows (PowerShell), in the project folder:")
print(f'  $env:EDA_OLLAMA_URL = "{TUNNEL_URL}"')
print(r"  .venv\Scripts\python.exe kaggle\health_check.py --judge")

## 6. Watchdog - leave this cell running (restarts dead processes; a new tunnel means a NEW URL)

In [ ]:
while True:
    if ollama_proc.poll() is not None:
        print(time.strftime("%H:%M:%S"), "ollama died -> restarting", flush=True)
        ollama_proc = start_ollama()
        time.sleep(10)
    if tunnel_proc.poll() is not None:
        tunnel_proc, TUNNEL_URL = start_tunnel()
        print(time.strftime("%H:%M:%S"), "NEW TUNNEL URL:", TUNNEL_URL, flush=True)
    print(time.strftime("%H:%M:%S"), "ollama:", bool(ollama_ready()), "| tunnel:", TUNNEL_URL, flush=True)
    time.sleep(300)

## Mode B (fallback): run everything in this notebook, no Docker

Use only when mode A is impossible. Results must be labelled `sandbox=subprocess` (no network/file isolation).

1. Upload the project folder (without `.venv/` and `runs/`) as a Kaggle Dataset, e.g. `eda-agent-code`, and
   add it to this notebook. **Do not regenerate the datasets on Kaggle**: use the uploaded files.
2. In a new cell (instead of cell 6):
   ```
   !cp -r /kaggle/input/eda-agent-code /kaggle/working/eda_agent
   %cd /kaggle/working/eda_agent
   !pip install -q -r requirements.txt
   ```
   This pins pandas 2.2.3 / numpy 2.1.3 / ... so results match the ground truth.
   **Restart the kernel** afterwards if pandas was already imported, then re-run cells 1-3.
3. Set the mode:
   ```
   import os
   os.environ["EDA_SANDBOX_MODE"] = "subprocess"
   os.environ["EDA_OLLAMA_URL"] = "http://localhost:11434"
   ```
4. `!python kaggle/health_check.py --url http://localhost:11434`, then run the eval (stage 5).